In [8]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# 2. Import  libraries
import torch
import torch.nn as nn
import torch.quantization
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import os

In [10]:
# 3. Set device and quantization engine
device = torch.device("cpu")

import torch.backends.quantized
torch.backends.quantized.engine = 'fbgemm'  # Works for x86 CPUs
print(f"Device: {device} | Quant Engine: {torch.backends.quantized.engine}")

Device: cpu | Quant Engine: fbgemm


In [11]:
# 4. Load CIFAR-10 (Test Set Only)
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

In [12]:

model_fp32 = models.mobilenet_v3_small(weights=None)
model_fp32.classifier[3] = nn.Linear(model_fp32.classifier[3].in_features, 10)

model_fp32.load_state_dict(torch.load(
    "/content/drive/MyDrive/AI_MODEL_OPTIMIZATION/models/mobilenetv3_cifar10_baseline.pth",
    map_location=device
))
model_fp32.eval()

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

In [13]:
# 6. Define accuracy evaluation function
def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"Accuracy: {acc:.2f}%")
    return acc

In [14]:
# 7. Evaluate original model
print("Evaluating FP32 model...")
acc_fp32 = evaluate(model_fp32)


Evaluating FP32 model...
Accuracy: 80.33%


In [15]:
# 8. Apply dynamic quantization
model_quantized = torch.quantization.quantize_dynamic(
    model_fp32,
    {nn.Linear},
    dtype=torch.qint8
)
print(" Model dynamically quantized.")

 Model dynamically quantized.


In [16]:
# 9. Evaluate quantized model
print("Evaluating quantized model...")
acc_int8 = evaluate(model_quantized)

Evaluating quantized model...
Accuracy: 80.29%


In [17]:
# 10. Save quantized model
quant_model_path = "/content/drive/MyDrive/AI_MODEL_OPTIMIZATION/models/mobilenetv3_quantized.pth"
torch.save(model_quantized.state_dict(), quant_model_path)
print(f" Quantized model saved to: {quant_model_path}")

 Quantized model saved to: /content/drive/MyDrive/AI_MODEL_OPTIMIZATION/models/mobilenetv3_quantized.pth


In [18]:
# 11. Compare model sizes
fp32_path = "/content/drive/MyDrive/AI_MODEL_OPTIMIZATION/models/mobilenetv3_cifar10_baseline.pth"
size_fp32 = os.path.getsize(fp32_path) / 1e6
size_int8 = os.path.getsize(quant_model_path) / 1e6

print(f"\n FP32 Size: {size_fp32:.2f} MB")
print(f" INT8 Size: {size_int8:.2f} MB")
print(f" Size Reduction: {(size_fp32 - size_int8) / size_fp32 * 100:.2f}%")


 FP32 Size: 6.25 MB
 INT8 Size: 4.45 MB
 Size Reduction: 28.86%
